# 1. Annotation Ingestion and Patch Visualization

This notebook demonstrates how to convert QuPath-exported GeoJSON annotations into a cached SQLite store, build class-specific patch extractors and visualise example patches for each class.

Modify the paths below to point to your own data. By default, the tiny dataset in `tests/test_data` is used.

In [ ]:
import sys
sys.path.append("../src")
from pathlib import Path
from tvs.io.annotations import geojson_to_sqlite
from tvs.data.extractors import build_filters, make_extractors
import matplotlib.pyplot as plt

# Paths (change these to use your own data)
data_dir = Path('../tests/test_data')
slide_path = data_dir / 'test_slide.svs'
geojson_path = data_dir / 'test_slide.geojson'
db_path = data_dir / 'test_slide.db'

# Convert GeoJSON to SQLiteStore
stats = geojson_to_sqlite(geojson_path, db_path, cache=False)
print('Annotation stats:', stats)

# Build class-specific patch extractors
filters = build_filters(['T', 'V'])
extractors = make_extractors(
    slide_path,
    str(db_path),
    filters,
    patch_size=512,
    stride=512,
    min_mask_ratio=0.5,
)

# Show a few example patches from each class
for label, ex in extractors.items():
    fig, axes = plt.subplots(1, 4, figsize=(8, 4))
    for i, ax in enumerate(axes):
        patch = ex[i]
        if isinstance(patch, tuple):
            patch = patch[0]
        ax.imshow(patch)
        ax.set_title(f'{label} #{i}')
        ax.axis('off')
    fig.suptitle(f'Example patches for class {label}')
    plt.tight_layout()
    plt.show()
